# Part 2 Deep Dive

This notebook summarizes the completed runs in `part_2/outputs`, compares the MNIST models, and explains how the Part 2 experiments evolved from a baseline CNN into architecture, augmentation, regularization, and hyperparameter comparisons.

The focus here is on the latest completed Part 2 runs from `2026-04-28`, not the older archived runs under the root-level `outputs/Part2` folder.

## Project Presentation View

A strong ML project report should make the experimental story easy to follow:

- what the prediction problem is
- what models were trained
- what design choices were compared
- what results were achieved
- which tradeoffs matter
- what should be improved next

For this Part 2 project, the clearest presentation is:

- problem: 10-class handwritten digit classification on MNIST
- data: official MNIST training and test splits, with a validation split carved out of the training data
- method: compare CNN capacity, regularization, data augmentation, and optimizer-level choices
- result: several CNN variants reach roughly `99%+` test accuracy, with the augmented batch-normalized tuning run producing the best saved result
- practical takeaway: once the model is a reasonable CNN, small training choices affect the last few tenths of a percent more than the basic ability to learn MNIST

## Elevator Pitch

Part 2 trains neural networks to classify MNIST digits from `0` to `9`. The main goal is not only to build a working classifier, but to show a controlled progression: baseline training, CNN architecture comparison, augmentation comparison, regularization comparison, and a larger hyperparameter sweep. The saved outputs show that CNNs are consistently strong on MNIST, but the best final result comes from combining a compact batch-normalized CNN with mild affine augmentation and weight decay.

## Scope

Included in the comparison:

- `run_2026-04-28_154540_cnn_medium`: latest standalone baseline run in `part_2/outputs`
- `augmentation_comparison_2026-04-28_154632`: paired run with and without training augmentation
- `cnn_comparison_2026-04-28_154826`: CNN capacity and architecture comparison
- `regularization_comparison_2026-04-28_155225`: dropout, batch normalization, L1, weight decay, and combined regularization comparison
- `hyperparameter_tuning_2026-04-28_155935`: broader tuning sweep, including the strongest saved Part 2 runs

Excluded from the main conclusions:

- older root-level archived outputs under `outputs/Part2`
- per-run generated report notebooks, because they are individual experiment reports rather than a cross-run summary
- smoke-test style folders and Python cache files

## Data And Split

The dataset used in Part 2 is MNIST. Each image is a single-channel `28 x 28` grayscale handwritten digit, and the label is one of ten classes: `0` through `9`.

The loader uses the official MNIST training split as the source for training and validation, and keeps the official MNIST test split separate for final evaluation.

For the completed runs summarized here, the data is divided like this:

- training set: `54000` images
- validation set: `6000` images
- test set: `10000` images

How this was produced:

- the official MNIST training split contains `60000` images
- `10%` of that split was used for validation
- the remaining `90%` was used for training
- the official MNIST test split was used only for final testing

The standard transform converts images to tensors and normalizes them with mean `0.5` and standard deviation `0.5`, which maps pixel values into roughly the `[-1, 1]` range. Augmented runs add random affine transformations to the training subset only.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path(r"C:\Users\emil_\vscode\Assignment1\part_2")
OUTPUT_ROOT = ROOT / "outputs"

RUN_GROUPS = {
    "augmentation": OUTPUT_ROOT / "augmentation_comparison_2026-04-28_154632",
    "cnn_architecture": OUTPUT_ROOT / "cnn_comparison_2026-04-28_154826",
    "regularization": OUTPUT_ROOT / "regularization_comparison_2026-04-28_155225",
    "hyperparameter_tuning": OUTPUT_ROOT / "hyperparameter_tuning_2026-04-28_155935",
    "standalone_baseline": OUTPUT_ROOT,
}

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def discover_runs():
    runs = []
    for summary_path in sorted(OUTPUT_ROOT.rglob("summary.json")):
        run_dir = summary_path.parent
        config_path = run_dir / "config.json"
        if not config_path.exists():
            continue
        group = "other"
        for group_name, group_path in RUN_GROUPS.items():
            try:
                run_dir.relative_to(group_path)
            except ValueError:
                continue
            group = group_name
            break
        runs.append((group, run_dir, summary_path, config_path))
    return runs

records = []
for group, run_dir, summary_path, config_path in discover_runs():
    summary = load_json(summary_path)
    config = load_json(config_path)
    records.append({
        "run": run_dir.name,
        "group": group,
        "model": config.get("model_name"),
        "conv_channels": tuple(config.get("conv_channels", [])),
        "num_conv_layers": config.get("num_conv_layers"),
        "kernel_size": config.get("kernel_size"),
        "hidden_size": config.get("classifier_hidden_size"),
        "activation": config.get("activation"),
        "dropout": config.get("dropout", 0.0),
        "batch_norm": config.get("batch_norm", False),
        "augmentation": config.get("augmentation_enabled", False),
        "learning_rate": config.get("learning_rate"),
        "weight_decay": config.get("weight_decay", 0.0),
        "l1_lambda": config.get("l1_lambda", 0.0),
        "input_noise_std": config.get("input_noise_std", 0.0),
        "adam_beta1": config.get("adam_beta1"),
        "adam_beta2": config.get("adam_beta2"),
        "adam_eps": config.get("adam_eps"),
        "batch_size": config.get("batch_size"),
        "epochs_budget": config.get("epochs"),
        "params": config.get("trainable_parameters"),
        "best_val_loss": summary.get("best_validation_loss"),
        "best_val_acc": summary.get("best_validation_accuracy"),
        "test_loss": summary.get("final_test_loss"),
        "test_acc": summary.get("final_test_accuracy"),
        "best_epoch": summary.get("best_epoch"),
        "epochs_completed": summary.get("epochs_completed"),
        "stopped_early": summary.get("stopped_early"),
        "time_to_best_s": summary.get("time_to_best_model_seconds"),
        "total_time_s": summary.get("total_training_time_seconds"),
        "avg_epoch_s": summary.get("average_epoch_time_seconds"),
        "run_dir": str(run_dir),
    })

df = pd.DataFrame(records)
df.sort_values(by="test_acc", ascending=False).reset_index(drop=True)

## What Each Model Is

### `cnn_small`

A compact two-convolution CNN with channels `[16, 32]`. It has the smallest parameter count in the latest architecture comparison and is useful as an efficiency baseline.

### `cnn_medium`

The main baseline CNN. It uses two convolution stages with channels `[32, 64]`, pooling after each convolution, and a fully connected classifier. The standalone baseline and augmentation comparison both build around this model.

### `cnn_dropout`

A medium CNN with dropout in the classifier. It tests whether randomly dropping hidden activations improves generalization.

### `cnn_deep_balanced`

A three-convolution CNN with channels `[32, 64, 64]` and a larger hidden layer. It tests depth without making the last convolution especially wide.

### `cnn_deep_wide`

A three-convolution CNN with channels `[32, 64, 128]`. It increases representational capacity in the final convolution stage and is the strongest model in the dedicated architecture comparison.

### `cnn_batchnorm` and `cnn_regularized`

`cnn_batchnorm` adds batch normalization and light dropout to a compact three-convolution CNN. `cnn_regularized` combines batch normalization with heavier dropout. These models are the main base for the tuning sweep.

## How The Experiments Are Set Up In This Project

Shared setup across the completed Part 2 runs:

- dataset task: MNIST digit classification, ten classes
- image size: `1 x 28 x 28`
- train/validation split: `90% / 10%` from the official training split
- test split: official MNIST test set
- loss: cross entropy
- optimizer: Adam
- checkpoint selection: best validation loss
- batch size: usually `64`
- seed: `42`
- device in saved runs: `cuda:0`

Main setup differences:

- architecture comparison changes model capacity, convolution depth, hidden size, dropout, and batch normalization.
- augmentation comparison keeps the medium CNN fixed and toggles random affine augmentation.
- regularization comparison tests dropout, batch normalization, weight decay, L1 penalty, and combined regularization.
- hyperparameter tuning combines architecture, augmentation, regularization, learning-rate, and Adam parameter variants.

In [ ]:
setup_cols = [
    "run",
    "group",
    "model",
    "conv_channels",
    "num_conv_layers",
    "kernel_size",
    "hidden_size",
    "activation",
    "dropout",
    "batch_norm",
    "augmentation",
    "learning_rate",
    "weight_decay",
    "l1_lambda",
    "params",
]
df[setup_cols].sort_values(by=["group", "model", "run"]).reset_index(drop=True)

## Performance Comparison

This section compares the completed runs by final test accuracy, validation accuracy, runtime, selected epoch, and parameter count.

The headline result is that the best runs are clustered tightly around `99%` test accuracy. On MNIST, that means the project is no longer asking whether the network can learn the task. The useful question becomes which small design choices give the best final accuracy without adding unnecessary runtime or model size.

## The Data Story

The raw results show a clear progression. A basic CNN is already strong on MNIST because the data is clean, centered, grayscale, and small. The architecture comparison confirms that increasing capacity helps, but the improvement is modest because the task is already well matched to convolutional models.

The augmentation comparison is more interesting. Mild affine augmentation improves the medium CNN from `99.13%` to `99.23%` in the paired comparison. In the broader tuning sweep, an augmented batch-normalized CNN reaches `99.39%`, the best saved Part 2 result.

Regularization does not behave as a simple more-is-better story. Dropout alone is useful in the regularization comparison, but the combined regularization run is not the strongest. This makes sense for MNIST: the data is not noisy enough to reward overly strong regularization, and too much constraint can slightly limit final accuracy.

The strongest conclusion is that the best Part 2 model is not the largest model. `tune_07_augmented` has about `206k` trainable parameters and beats wider or much larger variants. For MNIST, the best balance is a compact CNN with enough depth, batch normalization, mild augmentation, and light weight decay.

In [ ]:
perf_cols = [
    "run",
    "group",
    "model",
    "params",
    "best_val_acc",
    "test_acc",
    "test_loss",
    "best_epoch",
    "epochs_completed",
    "time_to_best_s",
    "total_time_s",
    "avg_epoch_s",
]
df[perf_cols].sort_values(by="test_acc", ascending=False).reset_index(drop=True)

In [ ]:
summary_lines = {
    "most_accurate": df.sort_values(by="test_acc", ascending=False).iloc[0],
    "lowest_test_loss": df.sort_values(by="test_loss", ascending=True).iloc[0],
    "quickest_total": df.sort_values(by="total_time_s", ascending=True).iloc[0],
    "quickest_per_epoch": df.sort_values(by="avg_epoch_s", ascending=True).iloc[0],
    "smallest_model": df.sort_values(by="params", ascending=True).iloc[0],
    "best_architecture_comparison": df[df["group"] == "cnn_architecture"].sort_values(by="test_acc", ascending=False).iloc[0],
    "best_regularization_comparison": df[df["group"] == "regularization"].sort_values(by="test_acc", ascending=False).iloc[0],
    "best_tuning_run": df[df["group"] == "hyperparameter_tuning"].sort_values(by="test_acc", ascending=False).iloc[0],
}

for label, row in summary_lines.items():
    print(
        f"{label}: {row['run']} | model={row['model']} | "
        f"test_acc={row['test_acc']:.4f} | val_acc={row['best_val_acc']:.4f} | "
        f"total_time_s={row['total_time_s']:.2f} | params={row['params']}"
    )

## Quickest Vs Most Accurate

### Most accurate

`tune_07_augmented` is the strongest saved Part 2 run. It reaches `99.39%` final test accuracy with a compact `cnn_batchnorm` model, mild random affine augmentation, and weight decay.

### Strongest architecture-only model

`cnn_deep_wide` is the best run in the dedicated CNN architecture comparison. It reaches `99.26%` test accuracy with channels `[32, 64, 128]`, showing that extra depth and a wider final convolution stage help compared with the smaller CNNs.

### Quickest training run

Among the discovered latest runs, `cnn_medium` from the CNN comparison has the shortest total training time at about `36.36s`, while still reaching `99.11%` test accuracy. That makes the plain medium CNN a strong practical baseline.

### Smallest model

`cnn_small` has the fewest parameters at about `105866`, but still reaches `98.80%` test accuracy. It is not the best final model, but it is a useful demonstration that MNIST does not require a large network.

## Hyperparameter Deep Dive

### Best run: `tune_07_augmented`

- model: `cnn_batchnorm`
- convolution channels: `[32, 64, 64]`
- batch normalization: enabled
- dropout: `0.1`
- augmentation: random affine with `10` degree rotation, translation up to `0.1`, scale range `0.95` to `1.05`
- optimizer: Adam
- learning rate: `1e-3`
- weight decay: `1e-4`
- trainable parameters: `206346`
- final test accuracy: `99.39%`

This run is a good final recommendation because it is accurate without being the largest model.

### Strong wide run: `tune_14_wide_3conv`

- model: custom wide three-convolution CNN
- trainable parameters: `451050`
- final test accuracy: `99.37%`
- augmentation: disabled

This run is almost as accurate as the best run, but it uses more than twice as many parameters.

### Deep augmented regularized run: `tune_10_deep_augmented`

- model: `cnn_regularized`
- batch normalization: enabled
- dropout: heavier than `cnn_batchnorm`
- augmentation: enabled
- final test accuracy: `99.30%`

This confirms that augmentation remains useful, but heavier regularization is not clearly better than the lighter batch-normalized setup.

### Architecture comparison winner: `cnn_deep_wide`

- channels: `[32, 64, 128]`
- final test accuracy: `99.26%`
- total runtime: about `39.84s`

This is the best result when only architecture capacity is varied.

In [ ]:
comparison_summary = (
    df.groupby("group")
    .agg(
        runs=("run", "count"),
        best_test_acc=("test_acc", "max"),
        median_test_acc=("test_acc", "median"),
        fastest_total_s=("total_time_s", "min"),
        smallest_params=("params", "min"),
    )
    .sort_values(by="best_test_acc", ascending=False)
)
comparison_summary

## Architecture Behavior

The dedicated CNN comparison shows that all CNN variants are strong, but the extra capacity of `cnn_deep_wide` gives the highest accuracy in that group. The small CNN is still respectable, which shows the task is simple enough for compact convolutional features.

The important point is not that larger is always better. The hyperparameter sweep shows a compact batch-normalized model can beat larger models once augmentation and regularization are tuned. Capacity matters, but training setup matters too.

In [ ]:
arch_df = df[df["group"] == "cnn_architecture"].copy()
arch_df[["run", "model", "conv_channels", "params", "test_acc", "best_val_acc", "total_time_s"]].sort_values(
    by="test_acc", ascending=False
).reset_index(drop=True)

## Augmentation And Regularization Behavior

The augmentation comparison is clean because it changes one main factor while keeping the baseline architecture fixed. The augmented version improves final test accuracy from `99.13%` to `99.23%`, which is small but meaningful at this accuracy level.

The regularization comparison is less linear. `dropout_only` is the strongest regularization-comparison run at `99.21%`, while `weight_decay_only` is weaker in that specific group. In the larger tuning sweep, however, weight decay combined with batch normalization and augmentation contributes to the best overall run.

The practical interpretation is that regularization should be tuned as a package. On MNIST, mild regularization and augmentation are useful, but heavy stacking of regularizers can slightly reduce peak accuracy.

In [ ]:
aug_reg_df = df[df["group"].isin(["augmentation", "regularization"])].copy()
aug_reg_df[[
    "run",
    "group",
    "model",
    "augmentation",
    "dropout",
    "batch_norm",
    "weight_decay",
    "l1_lambda",
    "test_acc",
    "best_val_acc",
    "total_time_s",
]].sort_values(by=["group", "test_acc"], ascending=[True, False]).reset_index(drop=True)

## Recommended Model Choices

### If the goal is the best final result

Choose `tune_07_augmented`. It gives the highest saved test accuracy in `part_2/outputs`.

### If the goal is the best simple baseline

Choose `cnn_medium`. It trains quickly, is easy to explain, and already reaches about `99.11%` test accuracy in the CNN comparison.

### If the goal is the best architecture-only comparison result

Choose `cnn_deep_wide`. It is the strongest model in the dedicated CNN architecture comparison.

### If the goal is the smallest defensible model

Choose `cnn_small`. It is much smaller than the other CNNs and still performs well, though it gives up some final accuracy.

### If the goal is the clearest assignment comparison

Use this comparison set:

- `cnn_small` as the compact baseline
- `cnn_medium` as the simple strong baseline
- `cnn_deep_wide` as the architecture-capacity winner
- `dropout_only` as the best dedicated regularization-comparison run
- `tune_07_augmented` as the final tuned recommendation

## Next Comparison Run Plan

Based on the completed Part 2 runs, the next clean comparison should reduce overlap and focus on the most meaningful contenders.

| model | epoch budget | reason |
|---|---:|---|
| `cnn_small` | `7` | Keep a compact reference point. |
| `cnn_medium` | `7` | Keep the simplest strong baseline. |
| `cnn_deep_wide` | `7` | Keep the best architecture-comparison model. |
| `cnn_batchnorm` with augmentation and weight decay | `10` | Rerun the current best setup with a slightly larger budget. |
| `cnn_regularized` with augmentation | `10` | Compare light vs heavier regularization under the same augmentation budget. |

A useful future improvement would be repeated runs with different seeds. The top results differ by only a few tenths of a percent, so seed variation could matter when deciding whether one configuration is truly better or just slightly luckier in one split.

## Highlight Performance Table

This table condenses the main comparison into one view. Green cells mark the best value for each decision metric: higher is better for accuracy, and lower is better for training time and parameter count.

In [ ]:
highlight_runs = [
    "tune_07_augmented",
    "tune_14_wide_3conv",
    "tune_10_deep_augmented",
    "cnn_deep_wide",
    "cnn_medium",
    "cnn_small",
    "dropout_only",
]
highlight_df = df[df["run"].isin(highlight_runs)].copy()
highlight_df = pd.DataFrame({
    "run": highlight_df["run"],
    "group": highlight_df["group"],
    "model": highlight_df["model"],
    "test_acc_%": highlight_df["test_acc"] * 100,
    "best_val_acc_%": highlight_df["best_val_acc"] * 100,
    "train_time_s": highlight_df["total_time_s"],
    "avg_epoch_s": highlight_df["avg_epoch_s"],
    "params_k": highlight_df["params"] / 1_000,
}).sort_values(by="test_acc_%", ascending=False).reset_index(drop=True)

try:
    styled_highlight_df = (
        highlight_df.style
        .format({
            "test_acc_%": "{:.2f}",
            "best_val_acc_%": "{:.2f}",
            "train_time_s": "{:.2f}",
            "avg_epoch_s": "{:.2f}",
            "params_k": "{:.1f}",
        })
        .highlight_max(subset=["test_acc_%", "best_val_acc_%"], color="#d8f5d0")
        .highlight_min(subset=["train_time_s", "avg_epoch_s", "params_k"], color="#d8f5d0")
        .set_caption("Key performance highlights across selected Part 2 MNIST models")
    )
except AttributeError as exc:
    print(f"Pandas styling unavailable: {exc}")
    styled_highlight_df = highlight_df

styled_highlight_df

## Main Conclusions

The highlight table above is the clearest final comparison because it puts the important tradeoffs side by side: predictive quality, training cost, and model size.

1. `tune_07_augmented` is the best saved Part 2 model by final test accuracy.
2. Mild affine augmentation is useful, especially when paired with batch normalization and light weight decay.
3. Bigger CNNs help, but the largest model is not necessary for the best result on MNIST.
4. Regularization works best when tuned carefully; stacking too much regularization is not automatically better.
5. For a final practical recommendation: use the augmented `cnn_batchnorm` setup for maximum accuracy, and use `cnn_medium` or `cnn_small` when simplicity and speed matter more than the last few tenths of a percent.